In [21]:
# from reddit import get_subreddit
# from crawl import crawl
# content_list = get_subreddit('web3')+get_subreddit('cardano')+crawl()
# print(len(content_list))

In [9]:
import pandas as pd
import os
import ast

# Specify the directory containing the CSV files
folder_path = 'formatted_result'

# List all CSV files in the directory
csv_files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

# Initialize an empty list to hold DataFrames
dataframes = []

# Loop through the list of CSV files and read them into DataFrames
for file in csv_files:
    file_path = os.path.join(folder_path, file)  # Create full file path
    print(file_path)
    df = pd.read_csv(file_path, sep = "|")  # Read the CSV file
    dataframes.append(df)  # Append the DataFrame to the list

# Concatenate all DataFrames into a single DataFrame
concatenated_df = pd.concat(dataframes, ignore_index=True)

# Display the first few rows of the concatenated DataFrame
print(concatenated_df.size)
print(concatenated_df['news_content'].to_list()[:10])
content_list = [str(title)+'\n'+str(content) for title, content in zip(concatenated_df['news_title'].to_list(), concatenated_df['news_content'].to_list())]
source_list = [source for source in concatenated_df['URL'].to_list()]
date_list = [date[:date.find(" ")] for date in concatenated_df["date"]]
coin_name_list = [ast.literal_eval(coin_name) for coin_name in concatenated_df["coins_name"]]
print(len(content_list))

formatted_result\formatted_result.csv
formatted_result\formatted_result_Cardano.csv
formatted_result\formatted_result_Tronix.csv
formatted_result\formatted_result_Web3.csv
6475
['This coupon content was created by the WIRED team—we select offers based on what our readers are shopping for and regularly check all posted content to make sure it does what it says it will, because we know how annoying broken promo codes are.\xa0When you buy something using these coupons, we may earn a small affiliate commission.Learn more 1051 new deals and promo codes uploaded this week by our Team Perhaps you\'ve seen the viral video explaining the history of Crocs and the 2006 movie Idiocracy, where they made their debut in a dystopian future where the dumb rule the dumber. As the story goes, the wardrobe designer needed silly and distinctive footwear for the Brawndo-swilling denizens to wear during their appearances before the Extreme Court and selected these bulbous foam clogs from a Colorado-based sta

In [23]:
# from openai import OpenAI

# client = OpenAI(api_key = "sk-sJAILfYY4hF8aVTM73A26fB09c834c7b8c41D4CeB652Fe95",base_url="https://openai.ss-gpt.com/v1")

# completion_list = [client.chat.completions.create(
#   model="gpt-4o",
#   messages=[
#     {"role": "user", "content": """You are a professional web3 analyst who browse forums and try to retrieve up-to-date insights. 
#      Please analysis the following web3 post and generate a analysis report solely base on the post.
#      """+content}
#   ]) for content in content_list]

# print(completion_list[0].choices[0].message.content)


In [10]:
# # Can do splitting here!
def process_text(text: str, source: str, date:str, coin_name:list) -> tuple:
    print("received text length:",len(text))
    doc = []
    for i in range(0, int(len(text)/1000)-1):
      doc.append((text[i*1000:(i+1)*1000], source,  date, coin_name))
    return doc

combined_list = []
for text, source, date,coin_name in zip(content_list,source_list,date_list, coin_name_list):
  combined_list += process_text(text, source, date, coin_name)

#combined_list = [(text, source, date,coin_name) for text, source, date,coin_name in zip(content_list,source_list,date_list, coin_name_list)]


received text length: 4667
received text length: 14396
received text length: 5337
received text length: 13865
received text length: 5160
received text length: 14564
received text length: 8500
received text length: 4935
received text length: 8800
received text length: 46551
received text length: 8921
received text length: 35602
received text length: 13102
received text length: 14607
received text length: 11183
received text length: 11956
received text length: 31432
received text length: 6587
received text length: 7090
received text length: 7030
received text length: 9119
received text length: 30381
received text length: 4624
received text length: 26478
received text length: 24016
received text length: 52727
received text length: 22579
received text length: 16121
received text length: 10109
received text length: 4074
received text length: 3025
received text length: 2945
received text length: 2953
received text length: 3009
received text length: 2917
received text length: 2958
received te

In [11]:
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
import os


combined_list = combined_list[:50]
texts = [item[0] for item in combined_list]  # The main content (text)
metadata = [
    {"source": item[1], "date": item[2], "coin_name": item[3]} for item in combined_list
]

# Set up OpenAI embeddings
embedding_model = OpenAIEmbeddings(
    base_url=os.getenv("OPENAI_BASE_URL"),
    openai_api_key=os.getenv("OPENAI_API_KEY")
)

# Create the FAISS index with metadata
faiss_index = FAISS.from_texts(texts, embedding_model, metadatas=metadata)

# Save the FAISS index locally
faiss_index.save_local("faiss_index_with_metadata")


print("FAISS index with metadata saved locally.")

FAISS index with metadata saved locally.


In [12]:
import os
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    base_url=os.getenv("OPENAI_BASE_URL"),
    openai_api_key=os.getenv("OPENAI_API_KEY")
)

# Load the FAISS index with metadata
loaded_faiss_index = FAISS.load_local("faiss_index_with_metadata", embedding_model, allow_dangerous_deserialization=True)

# Query the FAISS index
query = "Why is the sky blue?"
query_result = loaded_faiss_index.similarity_search(query, k=3)  # Retrieve top-3 matches

# Print the results
print("Query:", query)
print("Results:", query_result)
result_source = []
result_string = []
result_node = []
for i, result in enumerate(query_result):
  result_source.append(result.page_content)
  result_string.append(result.metadata['source'])

print((result_source, result_string, result_node))

Query: Why is the sky blue?
Results: [Document(metadata={'source': 'https://www.wired.com/story/user-owned-ai-illia-polosukhin-open-source-web3/', 'date': '2024-10-31', 'coin_name': ['Bitcoin', 'Market', 'Crypto']}, page_content='aving lunch in a Google café with a scientist named Illia Polosukhin. Born in Ukraine, Polosukhin had been at Google for nearly three years. He was assigned to the team providing answers to direct questions posed in the search field. It wasn’t going all that well. “To answer something on Google.com, you need something that’s very cheap and high-performing,” Polosukhin says. “Because you have milliseconds” to respond. When Polosukhin aired his complaints, Uszkoreit had no problem coming up with a remedy. “He suggested, why not use self-attention?” says Polosukhin. Polosukhin sometimes collaborated with a colleague named Ashish Vaswani. Born in India and raised mostly in the Middle East, he had gone to the University of Southern California to earn his doctorate 

In [11]:
combined_list[1]

("He Helped Invent Generative AI. Now He Wants to Save It | WIRED\nTo revisit this article, visit My Profile, thenView saved stories. In 2016, Google engineer Illia Polosukhin had lunch with a colleague, Jacob Uszkoreit. Polosukhin had been frustrated by a lack of progress in his project, using AI to provide useful answers to questions posed by users, and Uszkoreit suggested he try a technique he had been brainstorming that he called self-attention. Thus began an 8-person collaboration that ultimately resulted in a 2017 paper called “Attention Is All You Need,” whichintroduced the concept of transformersas a way to supercharge artificial intelligence. It changed the world. Eight years later, though, Polosukhin is not completely happy with the way things are shaking out. A big believer in open source, he’s concerned about the secretive nature of transformer-based large language models, even from companies founded on the basis of transparency. (Gee, who can that be?) We don’t know what t